In [32]:
#### Minnesota Vikings Bye Week Performance Analysis #####
# This is an analysis of Post Bye Week performance of the Minnesota Vikings
# The goal is to evaluate whether or not the first game after a bye week is an advantage or not
#
# Hypothesis: Since the Bye Week provides the team with an extra week of rest and preperation for their next game, the team should perform lead to more wins than loses
# Definition: To be considered an "advantage", the team should win at least 66% of games after a bye week, which is significantly higher than the 50% win rate expected by chance.
# How is a bye week defined? A BYE week is for this exercise is defined as any time the team has 13 or more days of rest in between games
#                            NOTE: We do not consider just 14 days as a bye week, as there are occasions when games are played on Thursday, Saturdays and Mondays that can throw off the 14 rest period.

In [33]:
# Import the necessary libraries
import nfl_data_py as nfl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from pathlib import Path

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)

# Cache a copy of the schedule data locally to avoid repeated downloads
path = Path('data') / 'full_schedule.csv'
if os.path.exists(path):
    print("Local copy of schedule data found. Loading from file...")
    schedules = pd.read_csv(path)
else:
    schedules = nfl.import_schedules(list(range(1999, 2025)))
    os.makedirs('.\\data', exist_ok=True)
    schedules.to_csv(path, index=False)
    print("Schedules downloaded and saved locally")

All libraries imported successfully!
Pandas version: 3.0.2
Local copy of schedule data found. Loading from file...


In [34]:
# Filter out the Vikings games from the rest of the schedule, and only include regular season games (No playoffs or pre season games)
vikingsSchedule = schedules[((schedules['home_team'] == 'MIN') | (schedules['away_team'] == 'MIN')) & (schedules['game_type'] == 'REG')]
print("Vikings schedule data loaded successfully!")

# Print and verify the schedule
print(vikingsSchedule.head())

Vikings schedule data loaded successfully!
            game_id  season game_type  week     gameday weekday gametime  \
0   1999_01_MIN_ATL    1999       REG     1  1999-09-12  Sunday      NaN   
23  1999_02_OAK_MIN    1999       REG     2  1999-09-19  Sunday      NaN   
33   1999_03_MIN_GB    1999       REG     3  1999-09-26  Sunday      NaN   
50   1999_04_TB_MIN    1999       REG     4  1999-10-03  Sunday      NaN   
65  1999_05_CHI_MIN    1999       REG     5  1999-10-10  Sunday      NaN   

   away_team  away_score home_team  ...  wind  away_qb_id  home_qb_id  \
0        MIN          17       ATL  ...   NaN  00-0003761  00-0002876   
23       OAK          22       MIN  ...   NaN  00-0005741  00-0003761   
33       MIN          20        GB  ...  20.0  00-0003761  00-0005106   
50        TB          14       MIN  ...   NaN  00-0004293  00-0003761   
65       CHI          24       MIN  ...   NaN  00-0010560  00-0003761   

          away_qb_name        home_qb_name    away_coach    h

In [39]:
# Not every column is needed. We will filter down to the columns that we need

neededCols = [
    'game_id',     # database ID of the game
    'season',      # Season(year) of the game
    'game_type',   # Regular season, pre season, or play offs
    'week',        # Week of the season (1-18 for regular season, 1-4 for pre season, 1-5 for play offs)
    'home_team',   # Home team abbreviation
    'away_team',   # Away team abbreviation
    'home_score',  # Home team score
    'away_score',  # Away team score
    'result',      # Result of the game (home win, away win, tie) If the result is POSITIVE, the home team won. If the result is NEGATIVE, the away team won. If the result is 0, the game was a tie.
    'home_rest',   # Number of days of rest for the home team before the game
    'away_rest',   # Number of days of rest for the away team before the game
    'div_game',    # Whether the game is a divisional game or not
    'location'     # Location of the game (home, away, neutral)
]

filteredVikingsSchedule = vikingsSchedule[neededCols]
print("Filtered schedule data to only include needed columns")

# Add in the differential for the Vikings (POSITIVE == Vikings won, NEGATIVE == Vikings lost, 0 == tie)
filteredVikingsSchedule['vikings_score_diff'] = np.where(filteredVikingsSchedule['home_team'] == 'MIN', filteredVikingsSchedule['home_score'] - filteredVikingsSchedule['away_score'], filteredVikingsSchedule['away_score'] - filteredVikingsSchedule['home_score'])

# Create a new column to indicate if the Vikings won the game or not (We do not use a straight bool for this to take ties into account)
filteredVikingsSchedule['vikings_win'] = filteredVikingsSchedule['vikings_score_diff'] > 0
filteredVikingsSchedule['vikings_tie'] = filteredVikingsSchedule['vikings_score_diff'] == 0
filteredVikingsSchedule['vikings_loss'] = filteredVikingsSchedule['vikings_score_diff'] < 0

# Create a new column to indicate if the game was after a bye week or not
filteredVikingsSchedule['had_bye_week'] = ((filteredVikingsSchedule['home_team'] == 'MIN') & (filteredVikingsSchedule['home_rest'] >= 13)) | ((filteredVikingsSchedule['away_team'] == 'MIN') & (filteredVikingsSchedule['away_rest'] >= 13))

# Print and verify the filtered schedule
print(filteredVikingsSchedule.head())

Filtered schedule data to only include needed columns
            game_id  season game_type  week home_team away_team  home_score  \
0   1999_01_MIN_ATL    1999       REG     1       ATL       MIN          14   
23  1999_02_OAK_MIN    1999       REG     2       MIN       OAK          17   
33   1999_03_MIN_GB    1999       REG     3        GB       MIN          23   
50   1999_04_TB_MIN    1999       REG     4       MIN        TB          21   
65  1999_05_CHI_MIN    1999       REG     5       MIN       CHI          22   

    away_score  result  home_rest  away_rest  div_game location  \
0           17      -3          7          7         0     Home   
23          22      -5          7          7         0     Home   
33          20       3          7          7         1     Home   
50          14       7          7          7         1     Home   
65          24      -2          7          7         1     Home   

    vikings_score_diff  vikings_win  vikings_tie  vikings_loss  had_

In [40]:
# Verify that we have the correct number of games in the schedule, and how many BYE games are in each season (should only be 1 per season)
print(filteredVikingsSchedule.groupby('season')['had_bye_week'].sum())

season
1999    1
2000    1
2001    2
2002    1
2003    1
2004    1
2005    1
2006    1
2007    1
2008    1
2009    1
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
Name: had_bye_week, dtype: int64


In [41]:
###### NOTE #######
# Every season has one bye week except 2001. This extra bye week due to the game being cancelled/moved due to the 9/11 attacks.
# We will include this bye week in the analysis, as it is still a bye game due to our definintion at the beginning of this notebook

In [44]:
# Now we look to see the total number of wins, losses and ties after a bye week.

# Overall Performance 1999-2024
totalGamesPlayed = len(filteredVikingsSchedule)
totalGamesWons = filteredVikingsSchedule['vikings_win'].sum()
totalGamesTied = filteredVikingsSchedule['vikings_tie'].sum()
totalGamesAfterBye = filteredVikingsSchedule['vikings_loss'].sum()
totalWinningPercentage = totalGamesWons / totalGamesPlayed

# Performance after a bye week
vikingsByeGames = filteredVikingsSchedule[filteredVikingsSchedule['had_bye_week'] == True]
totalGamesPlayedAfterBye = len(vikingsByeGames)
postByeWins = vikingsByeGames['vikings_win'].sum()
postByeTies = vikingsByeGames['vikings_tie'].sum()
postByeLosses = vikingsByeGames['vikings_loss'].sum()
postByeWinningPercentage = postByeWins / totalGamesPlayedAfterBye

print(f"Overall Performance (1999-2024):")
print(f"Total Games Played: {totalGamesPlayed}")
print(f"Total Games Won: {totalGamesWons}")
print(f"Total Games Tied: {totalGamesTied}")
print(f"Total Games Lost: {totalGamesAfterBye}")
print(f"Overall Winning Percentage: {totalWinningPercentage:.2%}")
print("-------------------------------")
print(f"Performance After Bye Week:")
print(f"Total Games Played: {totalGamesPlayedAfterBye}")
print(f"Total Games Won: {postByeWins}")
print(f"Total Games Tied: {postByeTies}")
print(f"Total Games Lost: {postByeLosses}")
print(f"Winning Percentage After Bye Week: {postByeWinningPercentage:.2%}")


Overall Performance (1999-2024):
Total Games Played: 420
Total Games Won: 224
Total Games Tied: 2
Total Games Lost: 194
Overall Winning Percentage: 53.33%
-------------------------------
Performance After Bye Week:
Total Games Played: 27
Total Games Won: 14
Total Games Tied: 0
Total Games Lost: 13
Winning Percentage After Bye Week: 51.85%


In [47]:
# We will now look at the point differential

avgPointDiffPostBye = vikingsByeGames['vikings_score_diff'].mean()
avgPointDiffPostByeWin = vikingsByeGames.loc[vikingsByeGames['vikings_win'] == True, 'vikings_score_diff'].mean()
avgPointDiffPostByeLoss = vikingsByeGames.loc[vikingsByeGames['vikings_loss'] == True, 'vikings_score_diff'].mean()
avgPointDiffPostByeTie = vikingsByeGames.loc[vikingsByeGames['vikings_tie'] == True, 'vikings_score_diff'].mean()
maxPointDiffPostByeWin = vikingsByeGames.loc[vikingsByeGames['vikings_win'] == True, 'vikings_score_diff'].max()
minPointDiffPostByeWin = vikingsByeGames.loc[vikingsByeGames['vikings_win'] == True, 'vikings_score_diff'].min()
maxPointDiffPostByeLoss = vikingsByeGames.loc[vikingsByeGames['vikings_loss'] == True, 'vikings_score_diff'].max()
minPointDiffPostByeLoss = vikingsByeGames.loc[vikingsByeGames['vikings_loss'] == True, 'vikings_score_diff'].min() 

print(f"Average Point Differential After Bye Week: {avgPointDiffPostBye:.2f}")
print("----- Post Bye Wins -----")
print(f"Average Point Differential After Bye Week Win: {avgPointDiffPostByeWin:.2f}")
print(f"Maximum Point Differential After Bye Week Win: {maxPointDiffPostByeWin:.2f}")
print(f"Minimum Point Differential After Bye Week Win: {minPointDiffPostByeWin:.2f}")
print("----- Post Bye Losses -----")
print(f"Average Point Differential After Bye Week Loss: {avgPointDiffPostByeLoss:.2f}")
print(f"Average Point Differential After Bye Week Tie: {avgPointDiffPostByeTie:.2f}")
print(f"Maximum Point Differential After Bye Week Loss: {maxPointDiffPostByeLoss:.2f}")
print(f"Minimum Point Differential After Bye Week Loss: {minPointDiffPostByeLoss:.2f}")

Average Point Differential After Bye Week: -2.89
----- Post Bye Wins -----
Average Point Differential After Bye Week Win: 8.00
Maximum Point Differential After Bye Week Win: 18.00
Minimum Point Differential After Bye Week Win: 3.00
----- Post Bye Losses -----
Average Point Differential After Bye Week Loss: -14.62
Average Point Differential After Bye Week Tie: nan
Maximum Point Differential After Bye Week Loss: -2.00
Minimum Point Differential After Bye Week Loss: -38.00


In [ ]:
# Get the point wins and point differentials for all non-bye games to compare to the bye week games
